In [0]:
-- 1. Span and basic shape
SELECT min(full_date) AS first_day, max(full_date) AS last_day, count(*) AS days,
       round(avg(net_sales)) AS avg_daily_sales, round(min(net_sales)) AS min_day, round(max(net_sales)) AS max_day,
       sum(CASE WHEN orders = 0 THEN 1 ELSE 0 END) AS zero_order_days
FROM ecom_dev.reporting.sales_daily;

-- 2. Month by month: trend and seasonality
SELECT year_month, round(sum(net_sales)) AS net_sales, sum(orders) AS orders, count(*) AS days
FROM ecom_dev.reporting.sales_daily
GROUP BY year_month ORDER BY year_month;

-- 3. Weekday pattern
SELECT day_name, round(avg(net_sales)) AS avg_sales, round(avg(orders)) AS avg_orders
FROM ecom_dev.reporting.sales_daily
GROUP BY day_name ORDER BY avg_sales DESC;

CREATE TABLE IF NOT EXISTS ecom_dev.ml.sales_forecast (
  forecast_run_id  STRING        NOT NULL,
  generated_at     TIMESTAMP     NOT NULL,
  model_name       STRING        NOT NULL,
  model_version    STRING,
  series           STRING        NOT NULL,
  forecast_date    DATE          NOT NULL,
  horizon_days     INT           NOT NULL,
  yhat             DECIMAL(18,2),
  yhat_lower       DECIMAL(18,2),
  yhat_upper       DECIMAL(18,2),
  interval_level   DOUBLE
) COMMENT 'Every forecast ever produced; append-only so accuracy can be tracked against actuals';

select * from ecom_dev.ml.sales_forecast;

ALTER TABLE ecom_dev.ml.sales_forecast ADD COLUMNS (grain STRING AFTER series);

WITH g AS (
  SELECT sum(CASE WHEN full_date >= '2026-10-04' THEN net_sales END)
       / sum(CASE WHEN full_date <= '2025-11-01' THEN net_sales END) AS growth
  FROM ecom_dev.reporting.sales_daily
  WHERE full_date BETWEEN '2026-10-04' AND '2026-10-31'
     OR full_date BETWEEN '2025-10-05' AND '2025-11-01'
),
ly AS (
  SELECT sum(CASE WHEN full_date BETWEEN '2025-11-02' AND '2025-12-01' THEN net_sales END) AS b1,
         sum(CASE WHEN full_date BETWEEN '2025-12-02' AND '2025-12-31' THEN net_sales END) AS b2,
         sum(CASE WHEN full_date BETWEEN '2026-01-01' AND '2026-01-30' THEN net_sales END) AS b3
  FROM ecom_dev.reporting.sales_daily
)
SELECT round(growth, 3) AS growth_vs_last_year,
       round(b1 * growth) AS block1_nov, round(b2 * growth) AS block2_dec, round(b3 * growth) AS block3_jan
FROM g CROSS JOIN ly;

UPDATE ecom_dev.ml.event_calendar
SET event_name = concat('festive_main_', year(start_date))
WHERE event_name = 'festive_main';

SELECT event_name, start_date, end_date, status
FROM ecom_dev.ml.event_calendar WHERE event_name LIKE 'festive%' ORDER BY start_date;

SELECT grain, forecast_date, horizon_days, round(yhat) AS forecast, round(yhat_lower) AS low, round(yhat_upper) AS high
FROM ecom_dev.ml.sales_forecast
WHERE grain = 'month'
  AND generated_at = (SELECT max(generated_at) FROM ecom_dev.ml.sales_forecast)
ORDER BY forecast_date;


-- 1. The current forecast: only the newest run, so Power BI never shows two forecasts at once
CREATE OR REPLACE VIEW ecom_dev.reporting.forecast_latest AS
SELECT grain, forecast_date, horizon_days,
       yhat AS forecast, yhat_lower AS forecast_low, yhat_upper AS forecast_high,
       interval_level, model_name, model_version, generated_at, forecast_run_id
FROM ecom_dev.ml.sales_forecast
WHERE forecast_run_id = (SELECT forecast_run_id FROM ecom_dev.ml.sales_forecast
                         ORDER BY generated_at DESC LIMIT 1);

-- 2. One daily line for the chart: actuals up to today, the forecast after, with its range
CREATE OR REPLACE VIEW ecom_dev.reporting.sales_actual_vs_forecast AS
SELECT coalesce(a.full_date, f.forecast_date) AS full_date,
       a.net_sales AS actual_sales,
       f.forecast, f.forecast_low, f.forecast_high
FROM ecom_dev.reporting.sales_daily a
FULL OUTER JOIN (SELECT forecast_date, forecast, forecast_low, forecast_high
                 FROM ecom_dev.reporting.forecast_latest WHERE grain = 'day') f
  ON f.forecast_date = a.full_date;

-- 3. Accuracy monitoring: EVERY past forecast against what actually happened
CREATE OR REPLACE VIEW ecom_dev.reporting.forecast_accuracy AS
SELECT f.forecast_run_id, f.model_version, f.generated_at, f.horizon_days, f.forecast_date,
       a.net_sales AS actual, f.yhat AS forecast, f.yhat_lower AS low, f.yhat_upper AS high,
       f.yhat - a.net_sales AS error,
       abs(f.yhat - a.net_sales) / nullif(a.net_sales, 0) AS abs_pct_error,
       a.net_sales BETWEEN f.yhat_lower AND f.yhat_upper AS inside_interval
FROM ecom_dev.ml.sales_forecast f
JOIN ecom_dev.reporting.sales_daily a ON a.full_date = f.forecast_date
WHERE f.grain = 'day';

